# 미국의 세탁기 무역조치와 한국 교역구조의 재편, 2010–2025 — 재현 노트북

논문 `논문_세탁기세이프가드.md`의 **표 11개와 그림 2장, 본문 인용 수치 전부**를 재생성한다.

- 자료: `data/processed/kcsdb.duckdb` (관세청 통관실적, 2007.01–2026.03)
- 품목: 완제품 = HS 8450110000·8450120000·8450190000·8450200000 / 부분품 = 8450900000
- 커널: **Python (kcsdb)** — 기본 python3에는 duckdb가 없다
- 산출: 각 표를 `outputs/`에 CSV로 저장
- **단위 규약(2026-08-21 변경)**: 노트북은 금액을 **백만 달러**로 계산하지만, 논문의 표와 본문은 **억 달러(소수 2자리)**로 적는다.
  변환은 `백만 / 100`이며 §12 끝의 셀이 논문에 붙일 억 달러 표를 그대로 찍어 준다. 반올림으로 합계가 어긋나는 행이 있으나 주석은 달지 않는다.

## 논문 구조와의 대응

| 이 노트북 | 산출 변수 | 논문 |
|---|---|---|
| §1 | `tab1` | 표 2 |
| §2 | `tab2` | 표 3 |
| §3 | `tab3` | (표 없음, IV.2절 본문 수치) |
| §4 | `tab4` | 표 4 |
| §5 | `tab5` | 표 5 |
| §6 | `tab6` | (표 없음, 그림 1의 원자료) |
| §6.1 | `frontrun` | 표 6 |
| §7 | `tab7` | 표 7 |
| §8 | `tab8` | 표 8 |
| §9 | `tab9`~`tab12` | 표 9~11 |

변수명 `tabN`과 `outputs/` 파일명은 옛 번호를 유지한다. 논문 표 번호와 어긋나므로 위 대응표로 읽을 것.

## 조치 연표 (포고문 원문 대조, 논문 §II.2)

| 시점 | 근거 | 내용 |
|---|---|---|
| 2018.02.07 | 포고 9694호(2018.01.23) | 세이프가드 발효. 완제품 쿼터 120만 대, 쿼터 내 20%/초과 50%. 부분품 5만 대 초과 50%. 3년 1일 시한 |
| 2018.10.22 | 미국 세관 | 1년차 쿼터 소진 |
| 2020.02.07 | 포고 9979호(2020.01.23) | 3년차부터 **연 120만 대를 분기당 30만 대로 균등 배분**(연초 몰아넣기 차단) |
| 2021.02.08 | 포고 10133호(2021.01.14) | **2년 연장**. 완제품 쿼터 내 15%→14%, 초과 35%→30%. 부분품 쿼터 11만→13만 대 |
| 2023.02.07 | — | **종료** |

대부분의 표는 조치 종료 후 궤적을 담기 위해 **2025년까지** 확장되어 있다.
`tab12`(논문 표 12)만 2015–2020으로 끊는데, 2021년 이후 CIF 수입 단가에 해운운임 급등이 섞이기 때문이다.

마지막 절(§12)이 논문 본문에 인용된 수치를 자동 검증한다. **본문 수정 시 이 절을 먼저 돌려 볼 것.**

수정본(2026-09-11)부터 논문 표 번호가 하나씩 밀렸다. 새 표 1(삼성전자와 LG전자의 대응 연표)은 미국 상무부 반덤핑 명령과 FHT의 문헌 자료로 만든 표라 이 노트북이 계산하지 않는다.


## 0. 설정

In [1]:
import os
import pandas as pd
import duckdb

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

# ---- 루트 자동 탐색 (저장소 어디서 열어도 동작) ----
ROOT = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(ROOT, "data", "processed", "kcsdb.duckdb")):
    up = os.path.dirname(ROOT)
    if up == ROOT:
        raise FileNotFoundError("KCSDB2 루트를 찾지 못했습니다")
    ROOT = up
DB = os.path.join(ROOT, "data", "processed", "kcsdb.duckdb")

TAB_DIR = "outputs"
os.makedirs(TAB_DIR, exist_ok=True)

con = duckdb.connect(DB, read_only=True)

# ---- 품목 정의 (논문 §III) ----
FIN_CODES = ('8450110000', '8450120000', '8450190000', '8450200000')
PRT_CODE  = '8450900000'
FIN = "hs10 IN ('8450110000','8450120000','8450190000','8450200000')"
PRT = "hs10 = '8450900000'"

# ---- 판매시장형 대조군 (논문 §IV.3에서 자료로 선별, §0에서는 결과만 상수화) ----
SALES_MKT = ('TW', 'SA', 'AU', 'CO', 'CL', 'TR', 'FR', 'PE', 'AE')
SALES_SQL = "('TW','SA','AU','CO','CL','TR','FR','PE','AE')"

def save(df, name):
    # 표를 CSV로 저장하고 그대로 돌려준다.
    df.to_csv(os.path.join(TAB_DIR, name), index=False, encoding='utf-8-sig')
    return df

print("DB:", DB)
print("기간:", con.execute("SELECT MIN(yyyymm), MAX(yyyymm) FROM fact_trade").fetchone())

DB: C:\Work\Projects\KCSDB2\data\processed\kcsdb.duckdb
기간: (200701, 202607)


## 1. HS 8450의 세번별 구성 (논문 표 2)

논문 §III. 2015–2019년 수출 누계.

In [2]:
tab1 = con.execute(f'''
SELECT f.hs10,
       COALESCE(d.name_ko, '(폐지코드)')                        AS 품목명,
       SUM(CASE WHEN f.stat_cd = 'US' THEN f.exp_dlr ELSE 0 END)/1e6 AS 대미,
       SUM(f.exp_dlr)/1e6                                       AS 전세계
FROM fact_trade f
LEFT JOIN dim_hs10 d USING (hs10)
WHERE f.hs10 LIKE '8450%' AND f.yyyymm BETWEEN 201501 AND 201912
GROUP BY 1, 2
ORDER BY 전세계 DESC
''').df()
save(tab1.round(1), 'tab1_hs10_composition.csv')

,hs10,품목명,대미,전세계
0,8450200000,1회의 세탁 능력이 건조한 섬유제품의 중량으로 10킬로그램을 초과하는 것,730.8,2145.0
1,8450900000,부분품,179.3,1854.9
2,8450110000,완전자동 세탁기,99.2,498.1
3,8450120000,그 밖의 세탁기(원심탈수기를 내장한 것으로 한정한다),0.6,1.9
4,8450190000,기타,0.1,1.2


## 2. 한국의 세탁기 수출 (논문 표 3) (완제품/부분품 × 대미/기타)

논문 §IV.1. 이 표가 "세이프가드는 최초 충격이 아니라 마지막 층"이라는 §IV.1의 근거다.

In [3]:
tab2 = con.execute(f'''
SELECT yyyymm//100 AS 연도,
       SUM(CASE WHEN {FIN} AND stat_cd =  'US' THEN exp_dlr ELSE 0 END)/1e6 AS 완제품_대미,
       SUM(CASE WHEN {FIN} AND stat_cd <> 'US' THEN exp_dlr ELSE 0 END)/1e6 AS 완제품_기타,
       SUM(CASE WHEN {PRT} AND stat_cd =  'US' THEN exp_dlr ELSE 0 END)/1e6 AS 부분품_대미,
       SUM(CASE WHEN {PRT} AND stat_cd <> 'US' THEN exp_dlr ELSE 0 END)/1e6 AS 부분품_기타
FROM fact_trade
WHERE hs10 LIKE '8450%' AND yyyymm//100 BETWEEN 2010 AND 2025
GROUP BY 1 ORDER BY 1
''').df()
tab2['연도'] = tab2['연도'].astype(int)
save(tab2.round(1), 'tab2_annual_exports.csv')

,연도,완제품_대미,완제품_기타,부분품_대미,부분품_기타
0,2010,638.7,758.6,82.8,242.0
1,2011,613.4,744.9,63.9,312.6
2,2012,525.3,823.9,58.8,289.3
3,2013,291.5,690.6,57.8,434.4
4,2014,147.5,638.3,46.2,541.8
5,2015,89.9,484.7,48.3,478.1
6,2016,153.6,402.5,45.3,469.5
7,2017,306.0,358.3,13.0,386.3
8,2018,155.7,296.1,21.9,182.8
9,2019,125.4,273.8,50.8,158.8


## 3. 집계 방식에 따른 차이 (논문 IV.2절 본문 수치)

논문 §IV.2의 핵심. 부분품을 합쳐 보면 대미·비미국 격차가 31.7%p에서 8.6%p로,
곧 4분의 1 남짓으로 희석된다(3.7배 차이).

In [4]:
def yoy(where, y0=2017, y1=2018):
    # 지정 품목군의 대미·비미국 전년 대비 변화율(%)
    r = con.execute(f'''
    SELECT yyyymm//100 AS yr,
           SUM(CASE WHEN stat_cd =  'US' THEN exp_dlr ELSE 0 END) AS us,
           SUM(CASE WHEN stat_cd <> 'US' THEN exp_dlr ELSE 0 END) AS row_
    FROM fact_trade WHERE {where} AND yyyymm//100 IN ({y0}, {y1})
    GROUP BY 1 ORDER BY 1
    ''').df().set_index('yr')
    us  = (r.loc[y1, 'us']   / r.loc[y0, 'us']   - 1) * 100
    row = (r.loc[y1, 'row_'] / r.loc[y0, 'row_'] - 1) * 100
    return us, row, us - row

tab3 = pd.DataFrame(
    [('HS 8450 전체', *yoy("hs10 LIKE '8450%'")),
     ('완제품만',      *yoy(FIN))],
    columns=['집계', '대미(%)', '비미국(%)', '차이(%p)'])
save(tab3.round(1), 'tab3_aggregation_effect.csv')

,집계,대미(%),비미국(%),차이(%p)
0,HS 8450 전체,-44.3,-35.7,-8.6
1,완제품만,-49.1,-17.4,-31.8


## 4. 목적지별 부분품 비중 (논문 표 4) (생산기지형 / 판매시장형 스크린)

논문 §IV.3. 60% 이상과 10% 이하로 갈리고, 그 사이에는 미국·캐나다·이집트 셋만 놓인다.
이 스크린이 §0의 `SALES_MKT` 상수를 정당화한다.
조치 종료 후 미국의 부분품 비중이 30.5%까지 오르는 것은 §5.1에서 따로 본다.

In [5]:
tab4 = con.execute(f'''
SELECT f.stat_cd                                                     AS 코드,
       COALESCE(c.name_ko_kcs, '?')                                  AS 국가,
       SUM(f.exp_dlr)/1e6                                            AS 수출액,
       SUM(CASE WHEN {PRT} THEN f.exp_dlr ELSE 0 END)
         / NULLIF(SUM(f.exp_dlr), 0) * 100                           AS 부분품비중
FROM fact_trade f
LEFT JOIN dim_country c USING (stat_cd)
WHERE f.hs10 LIKE '8450%' AND f.yyyymm//100 BETWEEN 2015 AND 2017
GROUP BY 1, 2
HAVING SUM(f.exp_dlr) > 3e7
ORDER BY 부분품비중 DESC
''').df()
tab4['유형'] = pd.cut(tab4['부분품비중'], [-0.1, 10, 50, 100],
                      labels=['판매시장형', '중간', '생산기지형'])
save(tab4.round(1), 'tab4_parts_share_screen.csv')

,코드,국가,수출액,부분품비중,유형
0,RU,러시아 연방,217.0,91.6,생산기지형
1,TH,태국,270.1,90.6,생산기지형
2,IN,인도,56.7,87.3,생산기지형
3,PL,폴란드,115.3,85.7,생산기지형
4,VN,베트남,115.8,83.9,생산기지형
5,MX,멕시코,254.8,72.5,생산기지형
6,CN,중국,389.1,68.9,생산기지형
7,IR,이란,148.0,61.1,생산기지형
8,BR,브라질,45.4,59.8,생산기지형
9,EG,이집트,51.8,35.6,중간


## 5. 부분품 수출의 목적지 (논문 표 5) (FHT 그림 1 패널 B의 독립 재현)

논문 §IV.4. 대중국 급증(2012→2014) → 급감(2017→2018), 베트남·태국의 부상.

In [6]:
tab5 = con.execute(f'''
SELECT yyyymm//100 AS 연도,
       SUM(CASE WHEN stat_cd = 'CN' THEN exp_dlr ELSE 0 END)/1e6 AS 중국,
       SUM(CASE WHEN stat_cd = 'TH' THEN exp_dlr ELSE 0 END)/1e6 AS 태국,
       SUM(CASE WHEN stat_cd = 'VN' THEN exp_dlr ELSE 0 END)/1e6 AS 베트남,
       SUM(CASE WHEN stat_cd = 'US' THEN exp_dlr ELSE 0 END)/1e6 AS 미국,
       SUM(CASE WHEN stat_cd = 'PL' THEN exp_dlr ELSE 0 END)/1e6 AS 폴란드,
       SUM(CASE WHEN stat_cd NOT IN ('CN','TH','VN','US','PL')
                THEN exp_dlr ELSE 0 END)/1e6                   AS 기타,
       SUM(exp_dlr)/1e6                                        AS 합계
FROM fact_trade
WHERE {PRT} AND yyyymm//100 BETWEEN 2010 AND 2025
GROUP BY 1 ORDER BY 1
''').df()
tab5['연도'] = tab5['연도'].astype(int)
save(tab5.round(1), 'tab5_parts_destinations.csv')

,연도,중국,태국,베트남,미국,폴란드,기타,합계
0,2010,24.8,38.7,1.3,82.8,1.0,176.2,324.8
1,2011,27.2,43.9,2.5,63.9,10.4,228.7,376.5
2,2012,71.5,32.2,2.9,58.8,9.4,173.2,348.1
3,2013,128.5,48.0,2.9,57.8,33.1,221.9,492.2
4,2014,170.7,60.8,0.9,46.2,37.9,271.5,588.0
5,2015,132.3,70.8,0.7,48.3,30.7,243.5,526.4
6,2016,89.3,80.4,30.5,45.3,45.3,224.0,514.8
7,2017,46.5,93.5,66.0,13.0,22.7,157.6,399.4
8,2018,20.7,57.7,29.5,21.9,6.5,68.4,204.7
9,2019,27.2,37.6,29.7,50.8,5.3,59.1,209.6


### 5.1 대미 부분품 비중의 이동 (조치 종료 후)

논문 §IV.4 후반. 2020년까지만 보면 "회복일 뿐 사상 최고가 아니다"였으나,
2025년까지 늘리면 대미 부분품이 관측 기간 최고치를 경신한다.
요점은 **부분품 총액이 아니라 그 안에서 미국이 차지하는 몫**이 늘었다는 것.

In [7]:
# 대미 부분품의 사상 최고 경신 시점과 총액 대비 위치
pre = tab5[tab5.연도 <= 2020]
i = pre['미국'].idxmax()
print(f"2020년까지 대미 부분품 최고: {pre.loc[i, '미국']:.1f} 백만$ ({int(pre.loc[i, '연도'])}년)")
print(f"2022년 이후 최고    : {tab5[tab5.연도 >= 2022]['미국'].max():.1f} 백만$")
print()
print(tab5.loc[tab5.연도.isin([2014, 2020, 2022, 2023, 2024, 2025]),
               ['연도', '미국', '합계']].to_string(index=False))

# 대미 HS8450 수출에서 부분품이 차지하는 비중 (3년 창)
WINDOWS = [('2015-2017', 201501, 201712), ('2018-2020', 201801, 202012),
           ('2021-2022', 202101, 202212), ('2023-2025', 202301, 202512)]
rows = []
for lab, a, b in WINDOWS:
    v = con.execute(f'''
    SELECT SUM(CASE WHEN {PRT} THEN exp_dlr ELSE 0 END) / NULLIF(SUM(exp_dlr), 0) * 100
    FROM fact_trade
    WHERE stat_cd = 'US' AND hs10 LIKE '8450%' AND yyyymm BETWEEN {a} AND {b}
    ''').fetchone()[0]
    rows.append((lab, v))
us_parts_share = pd.DataFrame(rows, columns=['기간', '대미_부분품비중_%'])
print()
print(us_parts_share.round(1).to_string(index=False))
save(us_parts_share.round(1), 'tab5b_us_parts_share.csv')

2020년까지 대미 부분품 최고: 82.8 백만$ (2010년)
2022년 이후 최고    : 133.0 백만$

  연도         미국         합계
2014  46.191949 587.984322
2020  53.116483 255.094706
2022  90.863927 250.913234
2023 133.015765 280.284426
2024 110.855737 263.704324
2025 132.647499 284.162978

       기간  대미_부분품비중_%
2015-2017        16.2
2018-2020        21.3
2021-2022        15.9
2023-2025        30.5


,기간,대미_부분품비중_%
0,2015-2017,16.2
1,2018-2020,21.3
2,2021-2022,15.9
3,2023-2025,30.5


## 6. 완제품 대미 수출 월별 (논문 그림 1의 원자료)

논문 §IV.5. 밀어내기(2017.11), 발효 후 급감, 쿼터 주기(2018.07–08 / 2019.01–03)가 모두 이 표에 있다.

In [8]:
tab6 = con.execute(f'''
SELECT yyyymm                                        AS 연월,
       SUM(exp_dlr)/1e6                              AS 금액_백만달러,
       SUM(exp_wgt)/1e6                              AS 중량_천톤,
       SUM(exp_dlr)/NULLIF(SUM(exp_wgt), 0)          AS 단가_USD_kg
FROM fact_trade
WHERE {FIN} AND stat_cd = 'US' AND yyyymm BETWEEN 201701 AND 201906
GROUP BY 1 ORDER BY 1
''').df()
save(tab6.round(2), 'tab6_monthly_us.csv')

,연월,금액_백만달러,중량_천톤,단가_USD_kg
0,201701,15.70,2.52,6.23
1,201702,20.14,3.17,6.36
2,201703,15.76,2.18,7.24
3,201704,21.90,2.91,7.52
4,201705,24.61,3.62,6.80
5,201706,21.88,3.28,6.66
6,201707,17.27,2.36,7.31
7,201708,15.55,2.07,7.51
8,201709,28.61,3.88,7.37
9,201710,40.36,5.52,7.32


### 6.1 밀어내기 규모 계산

논문 §IV.5 (가)·(나). 발효 전 초과분과 발효 후 부족분을 중량으로 비교한다.

In [9]:
def wgt(a, b):
    # 기간 [a, b]의 완제품 대미 수출 중량 합계(천 톤)와 개월 수
    v, n = con.execute(f'''
    SELECT SUM(exp_wgt)/1e6, COUNT(DISTINCT yyyymm) FROM fact_trade
    WHERE {FIN} AND stat_cd = 'US' AND yyyymm BETWEEN {a} AND {b}
    ''').fetchone()
    return v, n

base_v, base_n = wgt(201701, 201708)     # 정상 기준: 2017.01–08
base_m = base_v / base_n
surge_v, _     = wgt(201709, 201711)     # 밀어내기 구간
post_v, post_n = wgt(201801, 201806)     # 발효 후 6개월

excess   = surge_v - 3 * base_m          # 발효 전 초과분
shortage = (base_m - post_v / post_n) * post_n   # 발효 후 부족분

# 이 표가 논문 표 6다(2026-08-19 개정). 옛 30개월 월별표는 그림 1 2단 패널로 대체되었다.
frontrun = pd.DataFrame([
    ('2017.01-08 월평균 (정상 기준)',      round(base_m, 2)),
    ('2017.09-11 실적 합계',              round(surge_v, 2)),
    ('2017.09-11 정상 기대치 (2.76 x 3)', round(3 * base_m, 2)),
    ('밀어내기 초과분',                    round(excess, 2)),
    ('2018.01-06 월평균',                round(post_v / post_n, 2)),
    ('발효 후 부족분 (6개월 누계)',        round(shortage, 2)),
], columns=['구간', '중량_천톤'])
print(frontrun.to_string(index=False))
save(frontrun, 'tab6b_frontrunning.csv')

                          구간  중량_천톤
      2017.01-08 월평균 (정상 기준)   2.76
            2017.09-11 실적 합계  19.57
2017.09-11 정상 기대치 (2.76 x 3)   8.29
                    밀어내기 초과분  11.28
              2018.01-06 월평균   1.52
           발효 후 부족분 (6개월 누계)   7.47


,구간,중량_천톤
0,2017.01-08 월평균 (정상 기준),2.76
1,2017.09-11 실적 합계,19.57
2,2017.09-11 정상 기대치 (2.76 x 3),8.29
3,밀어내기 초과분,11.28
4,2018.01-06 월평균,1.52
5,발효 후 부족분 (6개월 누계),7.47


### 6.2 쿼터 주기의 소멸 (분기 배분 도입 이후)

논문 §IV.5 (라). 포고 9979호가 3년차(2020.02.07)부터 연 120만 대를 분기당 30만 대로 쪼갰다.
연초에 몰아넣어도 그 분기 한도를 넘으면 초과 세율을 물게 되므로 앞세우기 유인이 사라진다.

앞세우기가 두 형태로 나타났으므로 지표도 둘로 본다.

1. **1분기 집중도** = (1분기 월평균 중량) / (그해 월평균 중량)
   — 새 쿼터 연도 개시에 맞춘 앞세우기. **2019년만** 높다(2.35).
   2018년은 발효 직후 저점이라 오히려 0.64로 낮으니, "2018·2019년 1분기 급등"으로 뭉뚱그리면 틀린다.
2. **최대월 배율** = (연중 최대월 중량) / (그해 월평균 중량)
   — 쿼터 소진을 앞둔 몰아넣기(2018년 7~8월)를 잡는다.

두 지표 모두 분기 배분 이후 완만해져야 한다. 2020년은 팬데믹이 겹치므로 단독 근거로 쓰지 않는다.

In [10]:
# 완제품 대미 월별 중량, 2018–2025 (§6 월별표의 기간 확장판)
monthly = con.execute(f'''
SELECT yyyymm AS 연월,
       yyyymm//100 AS 연도,
       (yyyymm % 100 - 1)/3 + 1 AS 분기,
       SUM(exp_dlr)/1e6                     AS 금액_백만달러,
       SUM(exp_wgt)/1e6                     AS 중량_천톤,
       SUM(exp_dlr)/NULLIF(SUM(exp_wgt), 0) AS 단가_USD_kg
FROM fact_trade
WHERE {FIN} AND stat_cd = 'US' AND yyyymm BETWEEN 201801 AND 202512
GROUP BY 1 ORDER BY 1
''').df()
monthly[['연도', '분기']] = monthly[['연도', '분기']].astype(int)
save(monthly.round(2), 'tab6c_monthly_us_2018_2025.csv')

def season(g):
    mean = g['중량_천톤'].mean()
    return pd.Series({
        '1분기_집중도':  g.loc[g.분기 == 1, '중량_천톤'].mean() / mean,
        '최대월_배율':   g['중량_천톤'].max() / mean,
        '최대월':        int(g.loc[g['중량_천톤'].idxmax(), '연월'] % 100),
    })

q1 = monthly.groupby('연도').apply(season, include_groups=False).reset_index()
q1['최대월'] = q1['최대월'].astype(int)
# 쿼터 배분 방식: 3년차(2020.02.07)부터 분기 배분 — 2020년은 팬데믹 중첩
q1['쿼터_배분'] = ['연간 일괄' if y <= 2019 else '분기 배분' for y in q1['연도']]
save(q1.round(2), 'tab6d_q1_concentration.csv')
print(q1.round(2).to_string(index=False))
print()
print("2021·2022년 1~3월 중량(천 톤):")
print(monthly.loc[monthly.연월.isin([202101, 202102, 202103, 202201, 202202, 202203]),
                  ['연월', '중량_천톤']].round(2).to_string(index=False))

  연도  1분기_집중도  최대월_배율  최대월 쿼터_배분
2018     0.64    2.05    7 연간 일괄
2019     2.35    2.75    3 연간 일괄
2020     0.55    1.90   12 분기 배분
2021     0.72    1.60    7 분기 배분
2022     1.14    1.39    5 분기 배분
2023     1.24    1.54    4 분기 배분
2024     1.18    1.61    5 분기 배분
2025     1.27    1.44    5 분기 배분

2021·2022년 1~3월 중량(천 톤):
    연월  중량_천톤
202101   4.08
202102   3.79
202103   4.70
202201   4.12
202202   4.79
202203   4.76


## 7. 완제품 수출 단가 (논문 표 7) (대미 vs 판매시장형)

논문 §IV.6. 대미 프리미엄이 2017년 +21.0%에서 2018년 −1.1%로 사라진다.

In [11]:
tab7 = con.execute(f'''
SELECT yyyymm//100 AS 연도,
       SUM(CASE WHEN stat_cd =  'US' THEN exp_dlr ELSE 0 END)
         / NULLIF(SUM(CASE WHEN stat_cd =  'US' THEN exp_wgt ELSE 0 END), 0) AS 대미,
       SUM(CASE WHEN stat_cd IN {SALES_SQL} THEN exp_dlr ELSE 0 END)
         / NULLIF(SUM(CASE WHEN stat_cd IN {SALES_SQL} THEN exp_wgt ELSE 0 END), 0) AS 판매시장형,
       SUM(CASE WHEN stat_cd <> 'US' THEN exp_dlr ELSE 0 END)
         / NULLIF(SUM(CASE WHEN stat_cd <> 'US' THEN exp_wgt ELSE 0 END), 0) AS 전체기타
FROM fact_trade
WHERE {FIN} AND yyyymm//100 BETWEEN 2013 AND 2025
GROUP BY 1 ORDER BY 1
''').df()
tab7['연도'] = tab7['연도'].astype(int)
tab7['대미프리미엄_%'] = (tab7['대미'] / tab7['판매시장형'] - 1) * 100
# 조치 국면 표시 — 종료(2023.02) 후에도 프리미엄이 회복되지 않는 것이 §IV.6 후반의 논점
tab7['국면'] = pd.cut(tab7['연도'], [2012, 2017, 2022, 2025],
                      labels=['조치 전', '조치 중', '조치 후'])
save(tab7.round(2), 'tab7_unit_values.csv')

,연도,대미,판매시장형,전체기타,대미프리미엄_%,국면
0,2013,6.51,5.69,5.66,14.42,조치 전
1,2014,6.39,5.57,5.89,14.72,조치 전
2,2015,6.94,5.77,5.99,20.15,조치 전
3,2016,6.40,5.87,6.16,9.07,조치 전
4,2017,6.97,5.76,6.26,21.02,조치 전
5,2018,6.15,6.22,6.49,-1.10,조치 중
6,2019,6.30,6.41,6.55,-1.74,조치 중
7,2020,5.99,7.48,7.08,-19.93,조치 중
8,2021,7.01,8.35,7.58,-16.08,조치 중
9,2022,7.22,8.61,7.97,-16.10,조치 중


## 8. 완제품 수출 상위 목적지 (논문 표 8)

논문 §IV.7. 대미 감소분을 흡수한 시장이 없다 — 무역전환 부재의 근거.

In [12]:
# 목적지 집합·행 순서는 2012-2025 누계 상위 9개국(논문 표 8과 동일 범위).
# 2021년까지로 끊어도 아홉 나라 구성은 같고 순서만 달라진다(2026-08-21 확인).
YRS = [2012, 2015, 2017, 2018, 2019, 2021, 2023, 2025]
tops = [r[0] for r in con.execute(f'''
SELECT stat_cd FROM fact_trade
WHERE {FIN} AND yyyymm//100 BETWEEN 2012 AND 2025
GROUP BY 1 ORDER BY SUM(exp_dlr) DESC LIMIT 9
''').fetchall()]
names = dict(con.execute("SELECT stat_cd, name_ko_kcs FROM dim_country").fetchall())

wide = con.execute(f'''
SELECT stat_cd, yyyymm//100 AS yr, SUM(exp_dlr)/1e6 AS v
FROM fact_trade WHERE {FIN} AND yyyymm//100 IN ({','.join(map(str, YRS))})
GROUP BY 1, 2
''').df().pivot(index='stat_cd', columns='yr', values='v').fillna(0)

tab8 = wide.reindex(tops).fillna(0).round(1)
tab8.index = [names.get(c, c) for c in tops]
tab8.columns = [int(c) for c in tab8.columns]

# 2017->2018 비미국 합계 변화 (무역전환 점검)
row_1718 = con.execute(f'''
SELECT yyyymm//100 AS yr, SUM(exp_dlr)/1e6 AS v FROM fact_trade
WHERE {FIN} AND stat_cd <> 'US' AND yyyymm//100 IN (2017, 2018) GROUP BY 1 ORDER BY 1
''').df()
print("비미국 합계 2017 -> 2018 :",
      round(row_1718.v.iloc[1] - row_1718.v.iloc[0], 1), "백만 달러")
save(tab8.reset_index().rename(columns={'index': '목적지'}), 'tab8_destinations.csv')

비미국 합계 2017 -> 2018 : -62.2 백만 달러


,목적지,2012,2015,2017,2018,2019,2021,2023,2025
0,미국,525.3,89.9,306.0,155.7,125.4,487.3,291.4,233.3
1,대만,46.6,28.7,31.7,49.7,52.4,34.3,36.2,18.8
2,캐나다,99.7,5.9,15.7,13.9,12.5,66.3,52.6,49.0
3,멕시코,37.7,26.6,18.3,23.8,18.0,10.6,20.3,40.9
4,사우디아라비아,32.4,51.7,26.4,17.6,21.1,13.2,8.2,8.1
5,호주,52.9,41.9,25.1,13.2,9.6,5.4,7.1,5.8
6,중국,12.2,42.9,36.4,22.9,21.7,16.0,11.4,11.6
7,이란,109.6,20.2,20.3,2.0,0.0,0.0,0.0,0.0
8,콜롬비아,36.4,17.1,13.4,12.9,8.8,6.6,6.5,12.8


## 9. 수입측 조정 (논문 §V, 표 9~11)

FHT가 언급만 한 순수입국 전환을 수치로 확정하되, 2025년까지 늘리면
그 전환이 **2020년 한 해뿐**이었음도 함께 드러난다.
남는 사실은 순수출 규모가 2012년의 6.9% 수준으로 줄었다는 것.

이 절이 만드는 표는 넷이다.

| 변수 | 내용 | 논문 |
|---|---|---|
| `tab9` | 완제품 수출입·순수출 | 표 9 (§V.1) |
| `tab10` | 수입 원산지 | 표 10 (§V.2) |
| `tab11` | 대미 수출 중량 대 태국·베트남발 수입 중량 | 표 11 (§V.3) |
| `tab12` | 완제품 수입 단가 대 수출 단가 | 표 12 (§V.4) |

`tab12`(논문 표 12)가 §VI.2의 해석 문제에 직접 쓰인다. 한국이 들여오는 세탁기는 사실상 전부
삼성·LG 자신의 해외 공장에서 오는 기업내 거래이므로, 2018년에 이 단가가 어떻게
움직였는지가 "그룹 차원의 이전가격 개편"이라는 설명을 검증한다.

In [13]:
tab9 = con.execute(f'''
SELECT yyyymm//100 AS 연도,
       SUM(exp_dlr)/1e6 AS 수출,
       SUM(imp_dlr)/1e6 AS 수입,
       (SUM(exp_dlr) - SUM(imp_dlr))/1e6 AS 순수출
FROM fact_trade
WHERE {FIN} AND yyyymm//100 BETWEEN 2012 AND 2025
GROUP BY 1 ORDER BY 1
''').df()
tab9['연도'] = tab9['연도'].astype(int)
save(tab9.round(1), 'tab9_net_exports.csv')

# 순수입국이었던 해 / 순수출 축소 배율
neg = tab9.loc[tab9.순수출 < 0, '연도'].tolist()
ratio = tab9.loc[tab9.연도 == 2025, '순수출'].iloc[0] / tab9.loc[tab9.연도 == 2012, '순수출'].iloc[0]
print(f"순수입국이었던 해: {neg}")
print(f"순수출 2025 / 2012 = {ratio*100:.1f}%  (약 {1/ratio:.0f}분의 1)")

순수입국이었던 해: [2020]
순수출 2025 / 2012 = 6.9%  (약 15분의 1)


In [14]:
imp_wide = con.execute(f'''
SELECT stat_cd, yyyymm//100 AS yr, SUM(imp_dlr)/1e6 AS v
FROM fact_trade WHERE {FIN} AND yyyymm//100 IN ({','.join(map(str, YRS))})
GROUP BY 1, 2
''').df().pivot(index='stat_cd', columns='yr', values='v').fillna(0)

# 개별 행은 VN·TH·CN 셋뿐이다. 넷째 원산지부터는 규모가 없어 '기타'로 묶는다
# (표에 실린 여덟 해 누계로 4위 스페인 0.25억$ vs 1위 베트남 8.65억$).
_TOP3 = ['VN', 'TH', 'CN']
tab10 = imp_wide.loc[_TOP3].copy()
tab10.index = [names.get(c, c) for c in _TOP3]
tab10.loc['기타'] = imp_wide.drop(index=_TOP3).sum()
tab10.loc['합계'] = imp_wide.sum()
tab10 = tab10.round(1)
tab10.columns = [int(c) for c in tab10.columns]
save(tab10.reset_index().rename(columns={'index': '원산지'}), 'tab10_import_origins.csv')

# 세 나라 비중 추이 (§V.2 본문)
imp_share = (imp_wide.loc[_TOP3].sum() / imp_wide.sum() * 100).round(1)
print('3국 비중(%):', imp_share.to_dict())
print('기타 최대 (억$):', round(imp_wide.drop(index=_TOP3).sum().max() / 100, 2))

3국 비중(%): {2012: 79.3, 2015: 89.9, 2017: 93.2, 2018: 92.7, 2019: 94.3, 2021: 96.4, 2023: 98.2, 2025: 98.4}
기타 최대 (억$): 0.24


In [15]:
# ---- 논문 표 11 (§V.3): 대미 수출 중량 대 태국·베트남발 수입 중량 ----
# 두 흐름은 회계적으로 상쇄되는 관계가 아니다. 같은 두 기업의 같은 생산 배치 결정에서
# 나오는 두 결과라서 나란히 놓는다. 2018년에 비율이 1을 넘는 것이 §V.3의 논점.
tab11 = con.execute(f'''
SELECT yyyymm//100 AS 연도,
       SUM(CASE WHEN stat_cd = 'US' THEN exp_wgt END)/1e6              AS 대미수출_천톤,
       SUM(CASE WHEN stat_cd IN ('TH','VN') THEN imp_wgt END)/1e6      AS 태베수입_천톤
FROM fact_trade
WHERE {FIN} AND yyyymm//100 BETWEEN 2015 AND 2025
GROUP BY 1 ORDER BY 1
''').df()
tab11['연도'] = tab11['연도'].astype(int)
tab11['수입_수출_비율'] = tab11['태베수입_천톤'] / tab11['대미수출_천톤']
save(tab11.round(2), 'tab11_volume_crossover.csv')
print(tab11.round(2).to_string(index=False))

cross = tab11.loc[tab11['수입_수출_비율'] > 1, '연도'].min()
print(f"\n비율이 처음 1을 넘는 해: {int(cross)}년")
print(f"이후 1 아래로 내려간 해: "
      f"{tab11.loc[(tab11.연도 > cross) & (tab11['수입_수출_비율'] < 1), '연도'].tolist() or '없음'}")

  연도  대미수출_천톤  태베수입_천톤  수입_수출_비율
2015    12.97     7.95      0.61
2016    24.00    11.84      0.49
2017    43.91    30.41      0.69
2018    25.30    43.09      1.70
2019    19.92    58.48      2.94
2020    30.73    85.29      2.78
2021    69.56    84.71      1.22
2022    47.91    51.05      1.07
2023    42.10    68.93      1.64
2024    47.36    62.56      1.32
2025    38.41    70.29      1.83

비율이 처음 1을 넘는 해: 2018년
이후 1 아래로 내려간 해: 없음


In [16]:
# ---- 논문 표 12 (§V.4): 수입 단가 대 수출 단가 ----
# 핵심 논점: 2018년에 대미 수출 단가만 내리고, 판매시장형 수출도 기업내 수입도 올랐다.
# 수입은 CIF, 수출은 FOB이므로 수준이 아니라 변화율만 비교한다.
# 2021년 이후는 해운운임 급등이 CIF에 섞이므로 논문 표에서는 제외한다.
tab12 = con.execute(f'''
SELECT yyyymm//100 AS 연도,
  SUM(imp_dlr)/NULLIF(SUM(imp_wgt),0)                                                          AS 수입_전체,
  SUM(CASE WHEN stat_cd='TH' THEN imp_dlr END)/NULLIF(SUM(CASE WHEN stat_cd='TH' THEN imp_wgt END),0) AS 수입_태국,
  SUM(CASE WHEN stat_cd='VN' THEN imp_dlr END)/NULLIF(SUM(CASE WHEN stat_cd='VN' THEN imp_wgt END),0) AS 수입_베트남,
  SUM(CASE WHEN stat_cd='CN' THEN imp_dlr END)/NULLIF(SUM(CASE WHEN stat_cd='CN' THEN imp_wgt END),0) AS 수입_중국,
  SUM(CASE WHEN stat_cd='US' THEN exp_dlr END)/NULLIF(SUM(CASE WHEN stat_cd='US' THEN exp_wgt END),0) AS 수출_대미,
  SUM(CASE WHEN stat_cd IN {SALES_SQL} THEN exp_dlr END)
    /NULLIF(SUM(CASE WHEN stat_cd IN {SALES_SQL} THEN exp_wgt END),0)                          AS 수출_판매시장형
FROM fact_trade
WHERE {FIN} AND yyyymm//100 BETWEEN 2015 AND 2020
GROUP BY 1 ORDER BY 1
''').df()
tab12['연도'] = tab12['연도'].astype(int)
save(tab12.round(2), 'tab12_import_unit_values.csv')
print(tab12.round(2).to_string(index=False))

print("\n2017 -> 2018 변화율 (%)")
chg = (tab12.set_index('연도').loc[2018] / tab12.set_index('연도').loc[2017] - 1) * 100
print(chg.round(1).to_string())

# 발효(2018.02.07)를 낀 6개월 창 — 같은 달에 반대로 움직였는가
# 단가는 창 전체의 SUM(금액)/SUM(중량), 곧 가중 단가로 구한다(논문 표 7과 같은 방식).
# 월별 단가의 단순평균을 쓰면 물량이 적은 달이 과대 반영되어 값이 달라진다.
win = con.execute(f'''
SELECT CASE WHEN yyyymm BETWEEN 201708 AND 201801 THEN '발효전 6개월'
            ELSE '발효후 6개월' END AS 창,
  SUM(CASE WHEN stat_cd IN ('TH','VN') THEN imp_dlr END)
    /NULLIF(SUM(CASE WHEN stat_cd IN ('TH','VN') THEN imp_wgt END),0) AS 수입_태베,
  SUM(CASE WHEN stat_cd='US' THEN exp_dlr END)
    /NULLIF(SUM(CASE WHEN stat_cd='US' THEN exp_wgt END),0)           AS 수출_대미
FROM fact_trade
WHERE {FIN} AND yyyymm BETWEEN 201708 AND 201807
GROUP BY 1 ORDER BY 1
''').df()
save(win.round(2), 'tab12b_event_window_uv.csv')
print("\n발효 전후 6개월 창 (2017.08-2018.01 vs 2018.02-07)")
print(win.round(2).to_string(index=False))
for c in ['수입_태베', '수출_대미']:
    a, b = win[c].iloc[0], win[c].iloc[1]
    print(f"  {c}: {a:.2f} -> {b:.2f}  ({(b/a-1)*100:+.1f}%)")

  연도  수입_전체  수입_태국  수입_베트남  수입_중국  수출_대미  수출_판매시장형
2015   4.85   3.98    4.49   5.00   6.94      5.77
2016   4.58   3.71    5.13   4.60   6.40      5.87
2017   4.74   4.78    5.80   3.71   6.97      5.76
2018   4.88   5.17    5.45   3.41   6.15      6.22
2019   4.49   5.13    4.48   3.21   6.30      6.41
2020   3.94   4.48    3.68   3.32   5.99      7.48

2017 -> 2018 변화율 (%)
수입_전체        2.9
수입_태국        8.3
수입_베트남      -6.0
수입_중국       -8.0
수출_대미      -11.7
수출_판매시장형     8.0

발효 전후 6개월 창 (2017.08-2018.01 vs 2018.02-07)
      창  수입_태베  수출_대미
발효전 6개월   5.16   7.01
발효후 6개월   5.51   6.12
  수입_태베: 5.16 -> 5.51  (+6.8%)
  수출_대미: 7.01 -> 6.12  (-12.7%)


## 10. 자료 품질 점검

논문 §III의 중량 커버리지 주장과 8450.20 비중을 확인한다.

In [17]:
cov = con.execute(f'''
SELECT COUNT(*) AS 행수,
       SUM(CASE WHEN exp_wgt > 0 THEN 1 ELSE 0 END) AS 중량있는행,
       SUM(CASE WHEN exp_wgt > 0 THEN exp_dlr ELSE 0 END)
         / NULLIF(SUM(exp_dlr), 0) * 100 AS 금액커버리지
FROM fact_trade WHERE stat_cd = 'US' AND yyyymm//100 = 2025 AND exp_dlr > 0
''').df()
cov['행커버리지'] = cov['중량있는행'] / cov['행수'] * 100
print(cov.round(1).to_string(index=False))

share_2017 = con.execute(f'''
SELECT SUM(CASE WHEN hs10 = '8450200000' THEN exp_dlr ELSE 0 END)
         / NULLIF(SUM(exp_dlr), 0) * 100
FROM fact_trade WHERE {FIN} AND stat_cd = 'US' AND yyyymm//100 = 2017
''').fetchone()[0]
print(f"2017년 대미 완제품 중 8450200000 비중: {share_2017:.1f}%")

   행수   중량있는행  금액커버리지  행커버리지
52957 51836.0   100.0   97.9


2017년 대미 완제품 중 8450200000 비중: 85.9%


## 11. 그림

**원칙: 표로 전달되는 것은 그리지 않는다.** 표가 못 하는 일을 하는 그림만 남긴다.
연 단위 계열(논문 표 3·4·6·7·8·10·11)은 행이 6~16개라 표가 더 정확하고 조밀하므로 그림을 만들지 않는다.
계획서 §VII이 예정했던 4장과 개정 과정에서 늘어난 4장 가운데 6장을 이 원칙으로 잘랐다.
논문의 옛 표 7도 이 원칙에 따라 30개월 월별표에서 6행 계산표(현 표 6)로 줄었다. 월별 원자료는 그림 1이 대신한다.

남은 둘은 다음 이유로 표가 대체할 수 없다.

| 그림 | 무엇을 하는가 | 왜 표로는 안 되는가 |
|---|---|---|
| 1 | 완제품 대미 월별 물량·단가 2단 패널, 2017.01-2025.12 | **108개월 x 2계열**이라 표에 담기지 않는다. 위 패널은 §IV.5 (가)~(라)를, 아래 패널은 §IV.6의 단차 시점(2017.11)을 뒷받침한다. 단위가 달라 이중 축 대신 x축만 공유하는 2단 패널로 그렸다 |
| 2 | 월별 단가 지수 3계열, 2016.01-2019.12 | 논문 표 12는 **연 단위**라 하락이 언제 시작됐는지 말해 주지 못한다. 이 그림은 대미 단가만 발효 시점에 계단식으로 내려앉고 나머지 둘은 그러지 않는다는 **시점** 정보를 준다 |

그림 2가 지수(2017년 평균 = 100)인 것은 표현상의 선택이 아니라 규약의 강제다.
수출은 FOB, 수입은 CIF라 수준을 같은 축에 올릴 수 없으므로 변화율만 남긴다.

그림 표준(CLAUDE.md §3): Malgun Gothic, 흑백, 300dpi, top/right spine 제거, grid 0.87.
글리프 함정 때문에 라벨에는 ASCII 하이픈만 쓰고 `axes.unicode_minus = False`로 둔다.

In [18]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

DPI = 400          # 선명도: CLAUDE.md 표준 300에서 상향(이 논문 한정)

plt.rcParams.update({
    'font.family': 'Malgun Gothic',
    'axes.unicode_minus': False,      # U+2212 tofu 방지
    'font.size': 10,
    'figure.dpi': DPI,
})
IMG = 'img'
os.makedirs(IMG, exist_ok=True)

def style(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, color='0.87', linewidth=0.6)
    ax.set_axisbelow(True)

def xpos(ym):
    # yyyymm -> 연 단위 실수 좌표
    return ym // 100 + (ym % 100 - 1) / 12

# ================= 그림 1: 대미 완제품 월별 물량 (단일 패널) =================
# §IV.5 (가)~(라)를 뒷받침한다. 단가 패널은 2026-08-21 제거했다 —
# 같은 단차를 그림 2가 이미 그리고 있어 중복이었다(단가 수준의 연 단위 추이는 표 7).
g1 = con.execute(f"""
SELECT yyyymm,
       SUM(exp_wgt)/1e6                     AS wgt,
       SUM(exp_dlr)/NULLIF(SUM(exp_wgt), 0) AS uv
FROM fact_trade
WHERE {FIN} AND stat_cd = 'US' AND yyyymm BETWEEN 201701 AND 202512
GROUP BY 1 ORDER BY 1
""").df()
g1['x'] = g1.yyyymm.map(xpos)
assert len(g1) == 108, f'108개월이어야 한다: {len(g1)}'

EVENTS = [(2018 + 1/12,  '발효' + chr(10) + '2018.02'),
          (2018 + 9/12,  '쿼터 소진' + chr(10) + '2018.10'),
          (2020 + 1/12,  '분기 배분' + chr(10) + '2020.02'),
          (2023 + 1/12,  '종료' + chr(10) + '2023.02')]

fig, ax1 = plt.subplots(figsize=(9.2, 3.9))
ax1.plot(g1.x, g1.wgt, color='black', linewidth=1.15)
ax1.set_ylim(0, 11.6)
ax1.set_ylabel('중량 (천 톤)')
ax1.set_xlabel('연도')

for x, _ in EVENTS:
    ax1.axvline(x, color='0.45', linewidth=0.8, linestyle=(0, (4, 3)))
style(ax1)
for x, lab in EVENTS:
    ax1.text(x + 0.06, 11.35, lab, fontsize=9.75, va='top', ha='left',
             color='black', linespacing=1.2)
ax1.set_xlim(2016.9, 2026.05)
ax1.set_xticks(range(2017, 2026))
fig.tight_layout()
fig.savefig(f'{IMG}/wm_monthly_volume.png', dpi=DPI, bbox_inches='tight')
plt.close(fig)
print('그림 1 저장:', f'{IMG}/wm_monthly_volume.png', '(' + str(len(g1)) + '개월, 단일 패널)')

# ================= 그림 2: 월별 단가 지수 3계열 =================
g2 = con.execute(f"""
SELECT yyyymm,
  SUM(CASE WHEN stat_cd='US' THEN exp_dlr END)
    /NULLIF(SUM(CASE WHEN stat_cd='US' THEN exp_wgt END),0)             AS 대미수출,
  SUM(CASE WHEN stat_cd IN {SALES_SQL} THEN exp_dlr END)
    /NULLIF(SUM(CASE WHEN stat_cd IN {SALES_SQL} THEN exp_wgt END),0)   AS 판매시장형수출,
  SUM(CASE WHEN stat_cd IN ('TH','VN') THEN imp_dlr END)
    /NULLIF(SUM(CASE WHEN stat_cd IN ('TH','VN') THEN imp_wgt END),0)   AS 태베수입
FROM fact_trade
WHERE {FIN} AND yyyymm BETWEEN 201601 AND 201912
GROUP BY 1 ORDER BY 1
""").df()
assert g2.notna().all().all(), '월별 단가에 결측이 있다'
base = g2[(g2.yyyymm >= 201701) & (g2.yyyymm <= 201712)].mean(numeric_only=True)  # 2017 평균 = 100
g2['x'] = g2.yyyymm.map(xpos)

# 대미 수출을 초점 계열(검정·굵게)로, 나머지 둘을 대조 계열(회색)로 둔다.
SERIES = [('대미수출',      'black', '-',             1.7, '대미 수출 (FOB)'),
          ('판매시장형수출', '0.45',  (0, (5, 2)),     1.1, '판매시장형 수출 (FOB)'),
          ('태베수입',      '0.45',  (0, (1.5, 1.5)), 1.1, '태국·베트남 수입 (CIF)')]
fig, ax = plt.subplots(figsize=(8.6, 4.0))
for col, cl, ls, lw, lab in SERIES:
    ax.plot(g2.x, g2[col] / base[col] * 100, color=cl, linewidth=lw,
            linestyle=ls, label=lab)
ax.axhline(100, color='0.82', linewidth=0.7, zorder=0)
ax.axvline(2018 + 1/12, color='0.3', linewidth=0.9, linestyle=(0, (4, 3)))
ax.text(2018 + 1/12 + 0.05, 62, '세이프가드 발효 2018.02', fontsize=10.4,
        va='bottom', ha='left', color='black')
ax.set_xlim(2015.95, 2020.0)
ax.set_ylim(60, 152)
ax.set_xticks(range(2016, 2020))
ax.set_yticks(range(60, 141, 20))
ax.set_xlabel('연도')
ax.set_ylabel('단가 지수 (2017년 평균 = 100)')
ax.legend(frameon=False, fontsize=9.8, loc='upper left',
          bbox_to_anchor=(0.015, 0.99), ncol=1, labelspacing=0.55, handlelength=2.6)
style(ax)
fig.tight_layout()
fig.savefig(f'{IMG}/wm_uv_divergence.png', dpi=DPI, bbox_inches='tight')
plt.close(fig)
print('그림 2 저장:', f'{IMG}/wm_uv_divergence.png', f'({len(g2)}개월)')

# ---- 그림이 주장하는 바를 수치로 확인 ----
# 변수명에 fig2_ 접두를 붙인다. §12 검증 셀에 같은 이름(post)이 있어 덮어써진 적이 있다.
COLS = [s[0] for s in SERIES]
_ix = g2.set_index('yyyymm')[COLS]
fig2_pre  = _ix.loc[201708:201801].mean() / base[COLS] * 100
fig2_post = _ix.loc[201802:201807].mean() / base[COLS] * 100
print()
print('발효 전후 지수 (2017 평균 = 100)')
for col in COLS:
    print(f'  {col:8s}: {fig2_pre[col]:6.1f} -> {fig2_post[col]:6.1f}  ({fig2_post[col]-fig2_pre[col]:+.1f}p)')


그림 1 저장: img/wm_monthly_volume.png (108개월, 단일 패널)


그림 2 저장: img/wm_uv_divergence.png (48개월)

발효 전후 지수 (2017 평균 = 100)
  대미수출    :   99.6 ->   87.3  (-12.3p)
  판매시장형수출 :   96.7 ->  112.5  (+15.8p)
  태베수입    :  102.8 ->  110.1  (+7.4p)


## 12. 본문 인용 수치 검증

논문 본문에 문장으로 적힌 수치를 자동 대조한다.
**본문을 수정하기 전에 이 절을 실행해 `PASS`를 확인할 것.**
어긋나면 논문 수치 또는 여기 기대값 중 하나가 낡은 것이다.

`chk()`는 개별 수치를, `assert`는 논문의 서술이 의존하는 **정성적 조건**을 지킨다.
후자가 깨지면 수치만 고칠 게 아니라 해당 절을 다시 써야 한다.

- 순수입국이었던 해가 2020년 하나뿐 → §V.1·초록·결론
- 물량 비율이 2018년에 1을 넘고 이후 되돌아오지 않음 → §V.3·결론
- 2018년에 대미 수출 단가만 내리고 판매시장형·기업내 수입은 오름 → §V.4·§VI.2
- 조치 종료(2023) 후에도 대미 프리미엄이 음수 → §IV.6 후반·§VI.2 (가)
- 2019년 1분기 집중도 > 2.0, 분기 배분 이후 < 1.5 → §IV.5 (라)
- 2018년 최대월 = 7월 → §IV.5 (다)
- 단가 하락이 2017.11부터 시작 → §IV.6·§V.4
- 그림 2장이 `img/`에 존재하고 지수 대비가 유지됨 → §11

In [19]:
def pick(df, col, yr, key='연도'):
    return float(df.loc[df[key] == yr, col].iloc[0])

checks = []
def chk(label, got, want, tol=0.05):
    ok = abs(got - want) <= tol
    checks.append((label, round(got, 2), want, 'PASS' if ok else 'FAIL'))

# ---------- §IV.1 / 초록 ----------
chk('완제품 대미 2010 (백만$)',        pick(tab2, '완제품_대미', 2010), 638.7, 0.1)
chk('완제품 대미 2015 (백만$)',        pick(tab2, '완제품_대미', 2015),  89.9, 0.1)
chk('2010->2015 감소율 (%)',
    (pick(tab2, '완제품_대미', 2015) / pick(tab2, '완제품_대미', 2010) - 1) * 100, -85.9, 0.2)
chk('완제품 대미 2017 (백만$)',        pick(tab2, '완제품_대미', 2017), 306.0, 0.1)
chk('완제품 대미 2018 (백만$)',        pick(tab2, '완제품_대미', 2018), 155.7, 0.1)
chk('완제품 대미 2021 (백만$)',        pick(tab2, '완제품_대미', 2021), 487.3, 0.1)

# ---------- §IV.1 네번째·다섯번째 사실 (조치 종료 후) ----------
chk('완제품 대미 2022 (백만$)',        pick(tab2, '완제품_대미', 2022), 346.0, 0.1)
chk('완제품 대미 2023 (백만$)',        pick(tab2, '완제품_대미', 2023), 291.4, 0.1)
chk('완제품 대미 2024 (백만$)',        pick(tab2, '완제품_대미', 2024), 335.0, 0.1)
chk('완제품 대미 2025 (백만$)',        pick(tab2, '완제품_대미', 2025), 233.3, 0.1)
chk('완제품 대미 2024-25 평균 (백만$)',
    (pick(tab2, '완제품_대미', 2024) + pick(tab2, '완제품_대미', 2025)) / 2, 284.1, 0.1)
chk('부분품 대미 2023 (백만$)',        pick(tab2, '부분품_대미', 2023), 133.0, 0.1)
chk('부분품 대미 2025 (백만$)',        pick(tab2, '부분품_대미', 2025), 132.6, 0.1)
chk('부분품 대미 종전최고 2010 (백만$)', pick(tab2, '부분품_대미', 2010),  82.8, 0.1)

# ---------- §IV.2 집계 방식 대비 (논문 본문) ----------
chk('HS8450 전체 대미 변화 (%)',  tab3.loc[0, '대미(%)'],   -44.3)
chk('HS8450 전체 비미국 변화 (%)', tab3.loc[0, '비미국(%)'], -35.7)
chk('HS8450 전체 차이 (%p)',      tab3.loc[0, '차이(%p)'],   -8.6)
chk('완제품 대미 변화 (%)',        tab3.loc[1, '대미(%)'],   -49.1)
chk('완제품 비미국 변화 (%)',      tab3.loc[1, '비미국(%)'], -17.4)
# 표기 확정(2026-08-18): 반올림 전 값으로 계산한 -31.8%p로 본문·표를 통일했다.
# 표에 실린 -49.1과 -17.4를 빼면 -31.7이 나오지만, 본문 괄호주는 2026-08-20 삭제했다(표 7의 -11.7% 주석만 유지).
chk('완제품 차이 (%p) [정밀]',     tab3.loc[1, '차이(%p)'],  -31.8)
chk('희석 배율 (완제품차이/전체차이)',
    tab3.loc[1, '차이(%p)'] / tab3.loc[0, '차이(%p)'], 3.68, 0.02)

# ---------- §IV.2 희석의 산술 (2026-08-20 절 재작성으로 추가) ----------
def _pshare(yr, dest):
    """해당 연도·목적지 수출에서 부분품이 차지하는 비중(%)"""
    f = pick(tab2, f'완제품_{dest}', yr); p = pick(tab2, f'부분품_{dest}', yr)
    return p / (f + p) * 100

chk('2017 부분품비중 비미국 (%)', _pshare(2017, '기타'), 51.9, 0.1)
chk('2017 부분품비중 대미 (%)',   _pshare(2017, '대미'),  4.1, 0.1)
chk('부분품 비미국 2017->2018 (%)',
    (pick(tab2, '부분품_기타', 2018) / pick(tab2, '부분품_기타', 2017) - 1) * 100, -52.7, 0.1)

# 본문 2문단: 합산 시 이동폭 (대미 4.8%p vs 비미국 18.3%p)
chk('합산 이동폭 대미 (%p)',   tab3.loc[0, '대미(%)']   - tab3.loc[1, '대미(%)'],    4.8, 0.05)
chk('합산 이동폭 비미국 (%p)', tab3.loc[0, '비미국(%)'] - tab3.loc[1, '비미국(%)'], -18.3, 0.05)
# 본문 2문단: 비미국은 합산 이동폭이 대미보다 훨씬 커야 논증이 성립
assert abs(tab3.loc[0, '비미국(%)'] - tab3.loc[1, '비미국(%)'])        > 3 * abs(tab3.loc[0, '대미(%)'] - tab3.loc[1, '대미(%)']),     'IV.2절 2문단 재작성 필요: 비미국 쪽이 끌려 내려간다는 서술이 성립하지 않는다'
# 본문 3문단: "완제품 비미국이 -17.4%였으니 세 배가 넘는 낙폭이다"
assert ((pick(tab2, '부분품_기타', 2018) / pick(tab2, '부분품_기타', 2017) - 1) * 100)        / tab3.loc[1, '비미국(%)'] > 3.0,     'IV.2절 3문단 재작성 필요: 부분품 비미국 낙폭이 완제품 비미국의 세 배 미만'

# ---------- §IV.4 도입: 부분품은 공장으로 간다 (2026-08-21 추가) ----------
BASE_SQL = "('RU','TH','IN','PL','VN','MX','CN','IR','BR')"   # IV.3절 생산기지형 9개국
_pdest = con.execute(f'''
SELECT SUM(exp_dlr)                                                          AS tot,
       SUM(CASE WHEN stat_cd IN {SALES_SQL} THEN exp_dlr END)                AS mkt,
       SUM(CASE WHEN stat_cd IN {BASE_SQL}  THEN exp_dlr END)                AS base
FROM fact_trade WHERE {PRT} AND yyyymm BETWEEN 201501 AND 201712''').fetchone()
chk('부분품 -> 판매시장형 9개국 (%)', _pdest[1] / _pdest[0] * 100,  0.9, 0.05)
chk('부분품 -> 생산기지형 9개국 (%)', _pdest[2] / _pdest[0] * 100, 87.4, 0.05)
# 본문: "공장이 없는 곳으로도 조금은 흘러가지만 그 몫은 작다"
assert _pdest[1] / _pdest[0] < 0.02,     'IV.4절 도입 재작성 필요: 판매시장형으로 가는 부분품이 2%를 넘었다'

# ---------- §IV.4 대중국 하강과 대미 월별 (2026-08-21 절 재검토로 추가) ----------
_cn = con.execute(f'''SELECT yyyymm//100 y, SUM(exp_dlr)/1e6 v FROM fact_trade
 WHERE {PRT} AND stat_cd='CN' AND yyyymm//100 BETWEEN 2014 AND 2018 GROUP BY 1 ORDER BY 1''').df()
chk('대중국 부분품 2015 (백만$)', float(_cn.loc[_cn.y == 2015, 'v'].iloc[0]), 132.3, 0.1)
chk('대중국 부분품 2016 (백만$)', float(_cn.loc[_cn.y == 2016, 'v'].iloc[0]),  89.3, 0.1)
# 본문: "낙폭이 해마다 커져 ... 하강이 뚜렷이 가팔라진다"
_rate = _cn.v.pct_change().dropna().tolist()
assert all(b < a for a, b in zip(_rate, _rate[1:])),     f'IV.4절 재작성 필요: 대중국 낙폭이 해마다 커지지 않는다 {[round(r*100,1) for r in _rate]}'

_us = con.execute(f'''SELECT yyyymm, exp_dlr/1e6 v FROM fact_trade
 WHERE {PRT} AND stat_cd='US' AND yyyymm BETWEEN 201601 AND 201912''').df()
_pick = lambda ym: float(_us.loc[_us.yyyymm == ym, 'v'].iloc[0])
chk('대미 부분품 2016.11 (백만$)', _pick(201611), 1.5, 0.05)
chk('대미 부분품 2016.12 (백만$)', _pick(201612), 0.8, 0.05)
_h16 = _us[(_us.yyyymm >= 201601) & (_us.yyyymm <= 201610)].v
_y19 = _us[_us.yyyymm//100 == 2019].v
chk('2016.01-10 월평균 (백만$)', _h16.mean(), 4.3, 0.05)
chk('2019 월평균 (백만$)',       _y19.mean(), 4.2, 0.05)
# 본문: "2019년에는 다시 월 400만 달러 안팎이 된다" — 2016년 수준으로의 복귀
assert abs(_y19.mean() / _h16.mean() - 1) < 0.10,     'IV.4절 재작성 필요: 2019년 월평균이 2016년 1-10월 수준으로 돌아왔다고 볼 수 없다'
# 본문: "2016년 10월까지 월 400만~500만 달러"
assert _h16.max() < 6.0 and _h16.median() > 4.0,     'IV.4절 재작성 필요: 2016년 1-10월이 월 400만~500만 구간에 있지 않다'

# ---------- §IV.5 (라) 분기 내 쏠림 점검 (2026-08-21 추가) ----------
# 분기 배분은 앞세우기 유인을 없앤 것이 아니라 주기를 1년 -> 1분기로 줄인다.
# 그렇다면 분기 안에서 첫 달이 무거워야 하는데, 실제로는 반대다.
_qm = con.execute(f'''SELECT yyyymm//100 y, yyyymm%100 m, SUM(exp_wgt)/1e6 v
 FROM fact_trade WHERE {FIN} AND stat_cd='US' AND yyyymm BETWEEN 201802 AND 202512
 GROUP BY 1,2''').df()
_POS = {2:1, 3:2, 4:3, 5:1, 6:2, 7:3, 8:1, 9:2, 10:3, 11:1, 12:2, 1:3}   # 쿼터연도는 2월 시작
_qm['pos'] = _qm.m.map(_POS)
_qm['qy']  = _qm.apply(lambda r: r.y if r.m >= 2 else r.y - 1, axis=1)

def _relpos(lo, hi):
    g = _qm[(_qm.qy >= lo) & (_qm.qy <= hi)].copy()
    g['rel'] = g.v / g.groupby('qy').v.transform('mean')
    return g.groupby('pos').rel.mean()

_qsplit = _relpos(2021, 2022)      # 분기 배분기(팬데믹 2020 제외)
chk('분기 배분기 1개월차 (배)', _qsplit[1], 0.94, 0.005)
chk('분기 배분기 3개월차 (배)', _qsplit[3], 1.07, 0.005)
# 본문: "오히려 분기 뒤쪽이 무겁다" — 앞세우기가 분기 주기로 되살아나지 않았다
assert _qsplit[1] < _qsplit[3],     'IV.5절 (라) 재작성 필요: 분기 배분기에 분기 첫 달이 셋째 달보다 무겁다(분기 내 앞세우기 발생)'

# ---------- §V.2 원산지 집중 (2026-08-21 추가) ----------
chk('수입 3국 비중 2012 (%)', float(imp_share.loc[2012]), 79.3, 0.05)
chk('수입 3국 비중 2025 (%)', float(imp_share.loc[2025]), 98.4, 0.05)
chk('수입 기타 최대 (억$)',   imp_wide.drop(index=['VN','TH','CN']).sum().max() / 100, 0.24, 0.005)
# 본문: "2023년과 2025년에도 2017년 수준 아래로는 내려가지 않는다"
for _c in ['VN', 'TH', 'CN']:
    for _y in (2023, 2025):
        assert imp_wide.loc[_c, _y] >= imp_wide.loc[_c, 2017] - 1e-9,             f'V.2절 재작성 필요: {_c} {_y}년 수입이 2017년 아래로 내려갔다'
# 본문: 개별 행을 셋만 세운 근거 — 4위와의 격차
_cum = imp_wide.sum(axis=1).sort_values(ascending=False)
chk('수입 1위 누계 (억$)', _cum.iloc[0] / 100, 8.65, 0.005)
chk('수입 4위 누계 (억$)', _cum.iloc[3] / 100, 0.25, 0.005)

# ---------- §V.2 수요설 기각: 유럽 생산국은 제자리 (2026-08-21 추가) ----------
EU_SQL = "('ES','CZ','SE','DE','IT','PL','TR','SK')"      # 유럽 가전 생산국
_eu = con.execute(f'''SELECT yyyymm//100 y, SUM(imp_dlr)/1e8 v FROM fact_trade
 WHERE {FIN} AND stat_cd IN {EU_SQL} AND yyyymm//100 IN (2012, 2025) GROUP BY 1 ORDER BY 1''').df()
_tot = con.execute(f'''SELECT yyyymm//100 y, SUM(imp_dlr)/1e8 v FROM fact_trade
 WHERE {FIN} AND yyyymm//100 IN (2012, 2025) GROUP BY 1 ORDER BY 1''').df()
chk('유럽 생산국 수입 2012 (억$)', float(_eu.v.iloc[0]), 0.04, 0.005)
chk('유럽 생산국 수입 2025 (억$)', float(_eu.v.iloc[1]), 0.04, 0.005)
chk('전체 수입 2012->2025 (배)',   float(_tot.v.iloc[1] / _tot.v.iloc[0]), 10.5, 0.05)
# 본문: "전체 수입이 10.5배가 되는 동안 외국 브랜드의 유입은 늘지 않았다"
assert _eu.v.iloc[1] <= _eu.v.iloc[0] + 0.005,     'V.2절 재작성 필요: 유럽 생산국 수입이 2012년보다 늘었다(수요설 기각 근거가 무너짐)'

# ---------- §V.4 원산지별 수입 단가 (2026-08-21 추가) ----------
def _impuv(cond, lo, hi):
    return con.execute(f'''SELECT SUM(imp_dlr)/NULLIF(SUM(imp_wgt),0) FROM fact_trade
     WHERE {FIN} AND {cond} AND yyyymm BETWEEN {lo} AND {hi}''').fetchone()[0]

_g = lambda cond: (_impuv(cond, 201801, 201812) / _impuv(cond, 201701, 201712) - 1) * 100
_w = lambda cond: (_impuv(cond, 201802, 201807) / _impuv(cond, 201708, 201801) - 1) * 100

chk('수입단가 태국+베트남 2017->18 (%)', _g("stat_cd IN ('TH','VN')"),  4.3, 0.05)
chk('수입단가 중국 2017->18 (%)',        _g("stat_cd = 'CN'"),          -8.0, 0.05)
chk('수입단가 중국 6개월창 (%)',          _w("stat_cd = 'CN'"),          -8.7, 0.05)
# 본문: "이하 기업내 수입은 태국·베트남발을 가리킨다" — 이 계열은 두 창 모두에서 올라야 논증이 선다
assert _g("stat_cd IN ('TH','VN')") > 0 and _w("stat_cd IN ('TH','VN')") > 0,     'V.4절 재작성 필요: 태국·베트남발 수입 단가가 대미 수출과 같은 방향으로 움직였다'

# ---------- §III 물량 항목은 중량뿐 (2026-08-21 추가) ----------
_cols = {r[1] for r in con.execute("PRAGMA table_info('fact_trade')").fetchall()}
assert _cols == {'yyyymm','stat_cd','hs10','exp_dlr','imp_dlr','exp_wgt','imp_wgt','bal_payments'},     f'III장 재작성 필요: fact_trade 컬럼 구성이 바뀌었다 {sorted(_cols)}'
assert not any('qty' in c or 'cnt' in c for c in _cols),     'III장 재작성 필요: 대수 수량 컬럼이 생겼다 — "중량 단가일 수밖에 없다"를 고칠 것'
_u = dict(con.execute('''SELECT hs10, unit_qty FROM dim_hs10
 WHERE hs10 IN ('8450110000','8450120000','8450190000','8450200000','8450900000')''').fetchall())
assert all(_u[h] == 'U' for h in ['8450110000','8450120000','8450190000','8450200000']),     'III장 재작성 필요: 완제품 세번의 수량 단위가 대수(U)가 아니다'
assert _u['8450900000'] is None, 'III장 재작성 필요: 부분품에 수량 단위가 지정되었다'

# ---------- §III 중량 결측률 (2026-08-21 기준 정정) ----------
_w = con.execute(f'''SELECT COUNT(*) n, SUM(CASE WHEN exp_wgt > 0 THEN 1 ELSE 0 END) nw,
   SUM(exp_dlr) d, SUM(CASE WHEN exp_wgt > 0 THEN exp_dlr ELSE 0 END) dw
 FROM fact_trade WHERE {FIN} AND stat_cd = 'US' AND yyyymm//100 BETWEEN 2015 AND 2025 AND exp_dlr > 0''').fetchone()
chk('중량 있는 행 비중 (%)',   _w[1] / _w[0] * 100, 97.9, 0.05)
chk('중량 있는 금액 비중 (%)', _w[3] / _w[2] * 100, 100.0, 0.05)

# ---------- §IV.6 표 7 프리미엄 열 전수 (2026-08-23 추가) ----------
# 프리미엄은 반드시 반올림 전 단가로 계산한다. 2015·2016년이 반올림값 기준으로 잘못 적혀 있었다.
_uv = con.execute(f'''SELECT yyyymm//100 y,
  SUM(CASE WHEN stat_cd='US' THEN exp_dlr END)/NULLIF(SUM(CASE WHEN stat_cd='US' THEN exp_wgt END),0) us,
  SUM(CASE WHEN stat_cd IN {SALES_SQL} THEN exp_dlr END)
    /NULLIF(SUM(CASE WHEN stat_cd IN {SALES_SQL} THEN exp_wgt END),0) mk
 FROM fact_trade WHERE {FIN} AND yyyymm//100 BETWEEN 2013 AND 2025 GROUP BY 1 ORDER BY 1''').df().set_index('y')
_PREM = {2013: 14.4, 2014: 14.7, 2015: 20.1, 2016: 9.1, 2017: 21.0, 2018: -1.1, 2019: -1.7,
         2020: -19.9, 2021: -16.1, 2022: -16.1, 2023: -16.3, 2024: -13.1, 2025: -17.7}
for _y, _p in _PREM.items():
    chk(f'표7 {_y} 대미 프리미엄 (%)', (_uv.loc[_y, 'us'] / _uv.loc[_y, 'mk'] - 1) * 100, _p, 0.05)

# ---------- §IV.4 부분품 비중 이동 ----------
chk('대미 부분품비중 2015-17 (%)',
    float(us_parts_share.loc[us_parts_share.기간 == '2015-2017', '대미_부분품비중_%'].iloc[0]), 16.2, 0.1)
chk('대미 부분품비중 2023-25 (%)',
    float(us_parts_share.loc[us_parts_share.기간 == '2023-2025', '대미_부분품비중_%'].iloc[0]), 30.5, 0.1)
chk('부분품 총액 2014 정점 (백만$)', pick(tab5, '합계', 2014), 588.0, 0.2)
chk('부분품 총액 2023 (백만$)',      pick(tab5, '합계', 2023), 280.3, 0.2)
chk('부분품 총액 2025 (백만$)',      pick(tab5, '합계', 2025), 284.2, 0.2)

# ---------- §IV.4 논문 표 5 (2010-2025 전 연도 + 합계 열) ----------
assert tab5['연도'].tolist() == list(range(2010, 2026)),     "연도가 2010-2025 연속이 아니다 - 논문 표 5와 어긋난다"
for yr, cn, us, tot in [(2011, 27.2, 63.9, 376.5), (2013, 128.5, 57.8, 492.2),
                        (2015, 132.3, 48.3, 526.4), (2021, 30.7, 67.1, 271.5),
                        (2024, 25.3, 110.9, 263.7)]:
    chk(f'표5 {yr} 중국 (백만$)', pick(tab5, '중국', yr), cn, 0.1)
    chk(f'표5 {yr} 미국 (백만$)', pick(tab5, '미국', yr), us, 0.1)
    chk(f'표5 {yr} 합계 (백만$)', pick(tab5, '합계', yr), tot, 0.2)

# ---------- §IV.5 밀어내기 ----------
chk('2017.09-11 합계 (천톤)',  surge_v,     19.57, 0.02)
chk('밀어내기 초과분 (천톤)',   excess,      11.28, 0.02)
chk('밀어내기 배율 (실적/정상)', surge_v / (3 * base_m), 2.36, 0.02)
chk('2018 상반기 부족분 (천톤)', shortage,     7.47, 0.02)
chk('2017.11 중량 (천톤)',
    float(tab6.loc[tab6.연월 == 201711, '중량_천톤'].iloc[0]), 10.17, 0.02)

# ---------- §IV.5 (라) 분기 배분 이후 ----------
for ym, want in [(202101, 4.08), (202102, 3.79), (202103, 4.70),
                 (202201, 4.12), (202202, 4.79), (202203, 4.76)]:
    chk(f'{ym} 중량 (천톤)',
        float(monthly.loc[monthly.연월 == ym, '중량_천톤'].iloc[0]), want, 0.02)
# 계절 형태: 연간 일괄 배분기(2019)에는 1분기가 튀고, 분기 배분기에는 1 근방으로 내려앉는다.
# 2018년 1분기는 발효 직후 저점이라 오히려 낮다 — 그해의 앞세우기는 7~8월(최대월 배율)로 잡힌다.
qs = q1.set_index('연도')
chk('1분기 집중도 2019',  float(qs.loc[2019, '1분기_집중도']), 2.35, 0.02)
chk('1분기 집중도 2018',  float(qs.loc[2018, '1분기_집중도']), 0.64, 0.02)
chk('1분기 집중도 2021',  float(qs.loc[2021, '1분기_집중도']), 0.72, 0.02)
chk('1분기 집중도 2022',  float(qs.loc[2022, '1분기_집중도']), 1.14, 0.02)
chk('최대월 배율 2018',   float(qs.loc[2018, '최대월_배율']),  2.05, 0.01)
chk('최대월 배율 2019',   float(qs.loc[2019, '최대월_배율']),  2.75, 0.01)
chk('최대월 배율 2021',   float(qs.loc[2021, '최대월_배율']),  1.60, 0.02)
chk('최대월 배율 2022',   float(qs.loc[2022, '최대월_배율']),  1.39, 0.02)
assert qs.loc[2018, '최대월'] == 7, "2018년 최대월이 7월이 아니다 — §IV.5 (다)를 재확인할 것"
assert qs.loc[2019, '1분기_집중도'] > 2.0, "2019년 1분기 급등이 사라졌다 — §IV.5 (라)를 재작성할 것"
assert qs.loc[[2021, 2022], '1분기_집중도'].max() < 1.5,     "분기 배분 이후에도 1분기가 튄다 — §IV.5 (라)를 재작성할 것"

# ---------- §IV.6 단가 ----------
chk('대미 단가 2017',        pick(tab7, '대미', 2017),        6.97, 0.01)
chk('대미 단가 2018',        pick(tab7, '대미', 2018),        6.15, 0.01)
chk('판매시장형 단가 2017',   pick(tab7, '판매시장형', 2017),   5.76, 0.01)
chk('판매시장형 단가 2018',   pick(tab7, '판매시장형', 2018),   6.22, 0.01)
# 표기 확정: IV.2절과 같이 반올림 전 값 기준. 표의 6.97/6.15로 계산하면 -11.8이
# 나오므로 논문 표 7에 그 취지의 주를 달았다.
chk('대미 단가 변화 (%)',
    (pick(tab7, '대미', 2018) / pick(tab7, '대미', 2017) - 1) * 100, -11.7, 0.05)
chk('판매시장형 단가 변화 (%)',
    (pick(tab7, '판매시장형', 2018) / pick(tab7, '판매시장형', 2017) - 1) * 100, 8.0, 0.1)
chk('대미 프리미엄 2017 (%)', pick(tab7, '대미프리미엄_%', 2017), 21.0, 0.1)
chk('대미 프리미엄 2018 (%)', pick(tab7, '대미프리미엄_%', 2018), -1.1, 0.1)
# 조치 종료 후에도 프리미엄이 회복되지 않는다 — §IV.6 후반·§VI.2 (가)의 근거
chk('대미 프리미엄 2023 (%)', pick(tab7, '대미프리미엄_%', 2023), -16.3, 0.1)
chk('대미 프리미엄 2024 (%)', pick(tab7, '대미프리미엄_%', 2024), -13.1, 0.1)
chk('대미 프리미엄 2025 (%)', pick(tab7, '대미프리미엄_%', 2025), -17.7, 0.1)
prem_after = tab7.loc[tab7.연도 >= 2023, '대미프리미엄_%']
assert prem_after.max() < 0, "조치 종료 후 프리미엄이 양으로 돌아섰다 — §IV.6 후반을 재작성할 것"

# ---------- §V.1 순수출 ----------
chk('순수출 2012 (백만$)', pick(tab9, '순수출', 2012), 1313.6, 0.2)
chk('순수출 2020 (백만$)', pick(tab9, '순수출', 2020),  -82.3, 0.2)
chk('순수출 2025 (백만$)', pick(tab9, '순수출', 2025),   90.3, 0.2)
chk('순수출 2025/2012 (%)',
    pick(tab9, '순수출', 2025) / pick(tab9, '순수출', 2012) * 100, 6.9, 0.1)
assert tab9.loc[tab9.순수출 < 0, '연도'].tolist() == [2020], \
    "순수입국이었던 해가 2020년 하나가 아니다 — §V.1·초록·결론을 재작성할 것"

# ---------- §V.2 수입 원산지 (논문 표 10, 2025년까지 확장) ----------
imp_idx = tab10   # 국가명이 인덱스
for origin, yr, want in [('베트남', 2021, 215.0), ('베트남', 2023, 173.5), ('베트남', 2025, 162.2),
                         ('태국',  2023, 135.9), ('태국',  2025, 129.1),
                         ('중국',  2023,  86.9), ('중국',  2025,  77.2)]:
    chk(f'수입 {origin} {yr} (백만$)', float(imp_idx.loc[origin, yr]), want, 0.1)
# 세 원산지 모두 2021년 정점 뒤 줄지만 2017년보다는 높다 - §V.2의 "새 평형" 서술 근거
for origin in ['베트남', '태국', '중국']:
    assert imp_idx.loc[origin, 2025] < imp_idx.loc[origin, 2021], f'{origin}: 2021년이 정점이 아니다'
    assert imp_idx.loc[origin, 2025] > imp_idx.loc[origin, 2017], f'{origin}: 2025년이 2017년보다 낮다'

# ---------- §V.3 물량의 역전 ----------
for yr, w1, w2 in [(2015, 12.97, 7.95), (2017, 43.91, 30.41), (2018, 25.30, 43.09),
                   (2019, 19.92, 58.48), (2025, 38.41, 70.29)]:
    chk(f'{yr} 대미수출 중량 (천톤)', pick(tab11, '대미수출_천톤', yr), w1, 0.02)
    chk(f'{yr} 태베수입 중량 (천톤)', pick(tab11, '태베수입_천톤', yr), w2, 0.02)
chk('2018 수입/수출 비율', pick(tab11, '수입_수출_비율', 2018), 1.70, 0.01)
chk('2025 수입/수출 비율', pick(tab11, '수입_수출_비율', 2025), 1.83, 0.01)
assert pick(tab11, '수입_수출_비율', 2017) < 1 < pick(tab11, '수입_수출_비율', 2018),     "물량 역전이 2018년이 아니다 — §V.3을 재작성할 것"
assert (tab11.loc[tab11.연도 >= 2018, '수입_수출_비율'] > 1).all(),     "2018년 이후 비율이 1 아래로 되돌아간 해가 있다 — §V.3의 '되돌아오지 않는다'를 재작성할 것"

# ---------- §V.4 수입 단가 ----------
uv17, uv18 = tab12.set_index('연도').loc[2017], tab12.set_index('연도').loc[2018]
for col, want in [('수출_대미', -11.7), ('수출_판매시장형', 8.0),
                  ('수입_전체', 2.9), ('수입_태국', 8.3)]:
    chk(f'2018 {col} 변화 (%)', (uv18[col] / uv17[col] - 1) * 100, want, 0.15)
# 논문 §V.4의 핵심: 세 흐름 가운데 미국행만 내려갔다
assert uv18['수출_대미'] < uv17['수출_대미'], "대미 수출 단가가 내리지 않았다 — §V.4를 재작성할 것"
for col in ['수출_판매시장형', '수입_전체', '수입_태국']:
    assert uv18[col] > uv17[col], f"{col} 단가가 오르지 않았다 — §V.4의 대비 논증이 무너진다"
# 발효 전후 6개월 창에서도 같은 달에 반대로 움직였는가
a_i, b_i = win['수입_태베'].iloc[0], win['수입_태베'].iloc[1]
a_e, b_e = win['수출_대미'].iloc[0], win['수출_대미'].iloc[1]
chk('창: 대미 수출 단가 변화 (%)', (b_e / a_e - 1) * 100, -12.7, 0.1)
chk('창: 태베 수입 단가 변화 (%)', (b_i / a_i - 1) * 100, 6.8, 0.1)
assert b_e < a_e and b_i > a_i, "발효 전후 창에서 두 흐름이 반대로 움직이지 않았다 — §V.4를 재작성할 것"

# ---------- §IV.6 단가 하락의 시점 (그림 2가 드러낸 것) ----------
# 하락은 발효월(2018.02)이 아니라 밀어내기 절정인 2017.11부터 시작된다.
uvm = tab6.set_index('연월')['단가_USD_kg']
chk('2017.10 대미 단가', float(uvm[201710]), 7.32, 0.01)
chk('2017.11 대미 단가', float(uvm[201711]), 6.86, 0.01)
chk('2018.01 대미 단가', float(uvm[201801]), 6.13, 0.01)
assert uvm[201711] < uvm[201710] and uvm[201712] < uvm[201711],     "단가 하락이 2017.11부터 시작되지 않는다 - §IV.6·§V.4를 재작성할 것"

# ---------- §III 세번 비중 ----------
chk('2017 대미 8450.20 비중 (%)', share_2017, 86.0, 0.1)

# ---------- §III 완제품을 네 세번으로 묶는 근거 (2026-08-21 추가) ----------
FIN4_SQL = "('8450110000','8450120000','8450190000','8450200000')"
_t1 = con.execute(f'''SELECT hs10, SUM(exp_dlr) v FROM fact_trade
 WHERE hs10 IN {FIN4_SQL} AND stat_cd='US' AND yyyymm//100 BETWEEN 2015 AND 2019 GROUP BY 1''').df().set_index('hs10').v
chk('대미 완제품 중 8450200000 (%)',            _t1['8450200000'] / _t1.sum() * 100,             88.0, 0.05)
chk('대미 완제품 중 8450110000+200000 (%)', _t1[['8450110000','8450200000']].sum() / _t1.sum() * 100, 99.9, 0.05)

# 8450200000 하나로 좁혀도 IV.2·IV.6의 결론이 서는가
def _one(sel):
    q = con.execute(f'''SELECT yyyymm//100 y,
      SUM(CASE WHEN stat_cd='US' THEN exp_dlr END) us,
      SUM(CASE WHEN stat_cd<>'US' THEN exp_dlr END) nus,
      SUM(CASE WHEN stat_cd='US' THEN exp_dlr END)/NULLIF(SUM(CASE WHEN stat_cd='US' THEN exp_wgt END),0) uv_us,
      SUM(CASE WHEN stat_cd IN {SALES_SQL} THEN exp_dlr END)
        /NULLIF(SUM(CASE WHEN stat_cd IN {SALES_SQL} THEN exp_wgt END),0) uv_mkt
     FROM fact_trade WHERE hs10 IN {sel} AND yyyymm//100 IN (2017,2018) GROUP BY 1 ORDER BY 1''').df()
    g = lambda col: (q[col].iloc[1] / q[col].iloc[0] - 1) * 100
    return g('us') - g('nus'), g('uv_us'), g('uv_mkt')

_gap1, _uv1, _mkt1 = _one("('8450200000')")
chk('[8450200000만] 미국행-비미국행 격차 (%p)', _gap1, -31.8, 0.05)
chk('[8450200000만] 대미 단가 변화 (%)',        _uv1, -12.4, 0.05)
chk('[8450200000만] 판매시장형 단가 변화 (%)',   _mkt1,  +7.3, 0.05)
# 본문: "네 세번 기준과 거의 다르지 않다"
assert abs(_gap1 - tab3.loc[1, '차이(%p)']) < 0.5,     'III장 재작성 필요: 8450200000만으로 좁히면 IV.2절의 격차가 달라진다'


# ---------- 그림 (§11) ----------
for f in ['img/wm_monthly_volume.png', 'img/wm_uv_divergence.png']:
    assert os.path.exists(f), f"그림 파일이 없다: {f} - §11을 실행할 것"
for col, want in [('대미수출', 99.6), ('판매시장형수출', 96.7), ('태베수입', 102.8)]:
    chk(f'그림2 발효전 지수 {col}', float(fig2_pre[col]), want, 0.1)
for col, want in [('대미수출', 87.3), ('판매시장형수출', 112.5), ('태베수입', 110.1)]:
    chk(f'그림2 발효후 지수 {col}', float(fig2_post[col]), want, 0.1)
assert fig2_post['대미수출'] < fig2_pre['대미수출'], "그림 2: 대미 지수가 내리지 않았다"
assert (fig2_post[['판매시장형수출', '태베수입']] > fig2_pre[['판매시장형수출', '태베수입']]).all(),     "그림 2: 대조 계열이 오르지 않았다 - §V.4의 대비 논증이 무너진다"

# ================= to_eok로 재생성되지 않는 표의 전수 검증 (2026-08-23 추가) =================
# 표 4·5·10·11은 마지막 셀의 to_eok가 찍어 주지 않는다. 여기서 DB와 직접 대조해 둔다.

# ---------- 표 4. 목적지별 부분품 비중 (2015~2017 누계, 수출 0.3억 달러 초과국) ----------
_t3 = con.execute(f"""SELECT stat_cd, SUM(CASE WHEN {PRT} THEN exp_dlr ELSE 0 END) / SUM(exp_dlr) * 100 sh
     FROM fact_trade WHERE ({FIN} OR {PRT}) AND yyyymm BETWEEN 201501 AND 201712
     GROUP BY 1 HAVING SUM(exp_dlr) > 30000000""").df().set_index('stat_cd')['sh']
_T3 = {'RU': 91.6, 'TH': 90.6, 'IN': 87.3, 'PL': 85.7, 'VN': 83.9, 'MX': 72.5, 'CN': 68.9,
       'IR': 61.1, 'BR': 59.8, 'EG': 35.6, 'CA': 20.9, 'US': 16.2, 'AE': 9.0, 'TW': 2.4,
       'SA': 2.3, 'AU': 2.0, 'CO': 1.6, 'CL': 1.5, 'TR': 1.3, 'FR': 1.2, 'PE': 0.2}
assert set(_t3.index) == set(_T3), f"표 4 국가 집합 불일치: {set(_t3.index) ^ set(_T3)}"
for _c, _v in _T3.items():
    chk(f'표4 {_c} 부분품 비중 (%)', float(_t3[_c]), _v, 0.06)

# ---------- 표 6. 밀어내기와 그 반작용 (대미 완제품 중량, 천 톤) ----------
def _uswt(a, b):
    return con.execute(f"SELECT SUM(exp_wgt)/1e6 FROM fact_trade "
                       f"WHERE {FIN} AND stat_cd='US' AND yyyymm BETWEEN {a} AND {b}").fetchone()[0]
_base = _uswt(201701, 201708) / 8      # 정상 기준 월평균
_rush = _uswt(201709, 201711)          # 발효 전 3개월 실적
_h1   = _uswt(201801, 201806)          # 발효 후 6개월 실적
for _t, _a, _b in [('표6 정상 기준 월평균 (천톤)', _base, 2.76),
                   ('표6 2017.09~11 실적 (천톤)', _rush, 19.57),
                   ('표6 2017.09~11 기대치 (천톤)', _base * 3, 8.29),
                   ('표6 초과분 (천톤)', _rush - _base * 3, 11.28),
                   ('표6 2018.01~06 월평균 (천톤)', _h1 / 6, 1.52),
                   ('표6 2018.01~06 실적 (천톤)', _h1, 9.12),
                   ('표6 2018.01~06 기대치 (천톤)', _base * 6, 16.58),
                   ('표6 부족분 (천톤)', _base * 6 - _h1, 7.47)]:
    chk(_t, _a, _b, 0.006)
assert _rush - _base * 3 > _base * 6 - _h1, "표 6: 초과분이 부족분보다 크다는 IV.5절의 요점이 무너진다"

# ---------- 표 11. 대미 완제품 수출과 태국·베트남발 완제품 수입 (중량, 천 톤) ----------
_T10 = {2015: (12.97, 7.95, 0.61), 2016: (24.00, 11.84, 0.49), 2017: (43.91, 30.41, 0.69),
        2018: (25.30, 43.09, 1.70), 2019: (19.92, 58.48, 2.94), 2020: (30.73, 85.29, 2.78),
        2021: (69.56, 84.71, 1.22), 2022: (47.91, 51.05, 1.07), 2023: (42.10, 68.93, 1.64),
        2024: (47.36, 62.56, 1.32), 2025: (38.41, 70.29, 1.83)}
for _y, (_x, _m, _r) in _T10.items():
    _dx = _uswt(_y * 100 + 1, _y * 100 + 12)
    _dm = con.execute(f"SELECT SUM(imp_wgt)/1e6 FROM fact_trade WHERE {FIN} "
                      f"AND stat_cd IN ('TH','VN') AND yyyymm BETWEEN {_y*100+1} AND {_y*100+12}").fetchone()[0]
    chk(f'표11 {_y} 대미 수출 (천톤)', _dx, _x, 0.006)
    chk(f'표11 {_y} 태베 수입 (천톤)', _dm, _m, 0.006)
    chk(f'표11 {_y} 수입/수출', _dm / _dx, _r, 0.006)

# ---------- 표 12. 완제품 수입 단가와 수출 단가 (USD/kg) ----------
_T11 = {2015: (4.85, 3.98, 4.49, 5.00, 6.94, 5.77), 2016: (4.58, 3.71, 5.13, 4.60, 6.40, 5.87),
        2017: (4.74, 4.78, 5.80, 3.71, 6.97, 5.76), 2018: (4.88, 5.17, 5.45, 3.41, 6.15, 6.22),
        2019: (4.49, 5.13, 4.48, 3.21, 6.30, 6.41), 2020: (3.94, 4.48, 3.68, 3.32, 5.99, 7.48)}
for _y, _v in _T11.items():
    _q = con.execute(f"""SELECT SUM(imp_dlr)/NULLIF(SUM(imp_wgt),0),
        SUM(CASE WHEN stat_cd='TH' THEN imp_dlr END)/NULLIF(SUM(CASE WHEN stat_cd='TH' THEN imp_wgt END),0),
        SUM(CASE WHEN stat_cd='VN' THEN imp_dlr END)/NULLIF(SUM(CASE WHEN stat_cd='VN' THEN imp_wgt END),0),
        SUM(CASE WHEN stat_cd='CN' THEN imp_dlr END)/NULLIF(SUM(CASE WHEN stat_cd='CN' THEN imp_wgt END),0),
        SUM(CASE WHEN stat_cd='US' THEN exp_dlr END)/NULLIF(SUM(CASE WHEN stat_cd='US' THEN exp_wgt END),0),
        SUM(CASE WHEN stat_cd IN {SALES_SQL} THEN exp_dlr END)
          /NULLIF(SUM(CASE WHEN stat_cd IN {SALES_SQL} THEN exp_wgt END),0)
        FROM fact_trade WHERE {FIN} AND yyyymm BETWEEN {_y*100+1} AND {_y*100+12}""").fetchone()
    for _i, _nm in enumerate(['수입 전체', '태국', '베트남', '중국', '대미 수출', '판매시장형']):
        chk(f'표12 {_y} {_nm} (USD/kg)', _q[_i], _v[_i], 0.006)

# ---------- §IV.4 대미 공급지 이동 시점 (수정본 2026-09-11) ----------
# 논문: "대베트남 부분품 수출도 2016년에 처음 증가한다" → 2016년 중국산 반덤핑에 대한 대응으로 본다
_vn = con.execute(f"""SELECT yyyymm//100 y, SUM(exp_dlr)/1e8 v FROM fact_trade
     WHERE {PRT} AND stat_cd='VN' AND yyyymm BETWEEN 201401 AND 201712 GROUP BY 1""").df().set_index('y')['v']
assert _vn[2014] < 0.05 and _vn[2015] < 0.05 and _vn[2016] > 0.25, \
    "대베트남 부분품이 2016년에 처음 증가한다는 IV.4절 서술이 무너진다 - 수정본 IV.4절을 다시 쓸 것"

result = pd.DataFrame(checks, columns=['항목', '계산값', '논문값', '판정'])
print(result.to_string(index=False))
n_fail = (result['판정'] == 'FAIL').sum()
print()
print(f"=> {len(result)}건 중 FAIL {n_fail}건")
assert n_fail == 0, "논문 수치와 불일치. 위 표의 FAIL 항목을 확인할 것."

                            항목     계산값     논문값   판정
             완제품 대미 2010 (백만$)  638.71  638.70 PASS
             완제품 대미 2015 (백만$)   89.94   89.90 PASS
            2010->2015 감소율 (%)  -85.92  -85.90 PASS
             완제품 대미 2017 (백만$)  306.05  306.00 PASS
             완제품 대미 2018 (백만$)  155.71  155.70 PASS
             완제품 대미 2021 (백만$)  487.31  487.30 PASS
             완제품 대미 2022 (백만$)  346.04  346.00 PASS
             완제품 대미 2023 (백만$)  291.44  291.40 PASS
             완제품 대미 2024 (백만$)  335.01  335.00 PASS
             완제품 대미 2025 (백만$)  233.27  233.30 PASS
       완제품 대미 2024-25 평균 (백만$)  284.14  284.10 PASS
             부분품 대미 2023 (백만$)  133.02  133.00 PASS
             부분품 대미 2025 (백만$)  132.65  132.60 PASS
        부분품 대미 종전최고 2010 (백만$)   82.78   82.80 PASS
           HS8450 전체 대미 변화 (%)  -44.33  -44.30 PASS
          HS8450 전체 비미국 변화 (%)  -35.69  -35.70 PASS
             HS8450 전체 차이 (%p)   -8.64   -8.60 PASS
                 완제품 대미 변화 (%)  -49.12  -49.10 PASS
            

---

**미완**: 그림은 두 장만 만들었다(§11). "표로 전달되는 것은 그리지 않는다"는 원칙에 따라
계획서 §VII의 4장과 개정 과정에서 늘어난 4장 가운데 6장을 잘랐다. 연 단위 계열은 행이 6~16개라
표가 더 정확하고 조밀하다. 남긴 둘은 각각 108개월·48개월 계열이라 표로 옮길 수 없다.

**남은 과제**
- DOCX·영문판 미작성 (따라서 4파일 동시 수정 규칙은 아직 적용 대상 아님)
- 포고 9694호·9979호 원문 미대조 (10133호와 부속서는 대조 완료)
- Irwin(2019)·Cavallo et al.(2021)은 본문 미인용 상태로 참고문헌에 남아 있음


In [20]:
# ---------- 논문용 억 달러 표 재생성 (원본에서 한 번만 반올림) ----------
def to_eok(df, keep=1, signed=()):
    """금액 열을 억 달러(소수 2자리)로 바꾼 마크다운 표를 찍는다. keep = 변환하지 않을 앞 열 수."""
    out = df.copy()
    if not isinstance(out.index, pd.RangeIndex):
        out = out.reset_index()
    for c in out.columns[keep:]:
        if out[c].dtype.kind in 'if':
            out[c] = out[c].map(lambda v: ('+' if (c in signed and v > 0) else '') + f'{v/100:.2f}')
    print('| ' + ' | '.join(map(str, out.columns)) + ' |')
    print('|' + '---|' * len(out.columns))
    for _, r in out.iterrows():
        print('| ' + ' | '.join(map(str, r.values)) + ' |')
    print()

for label, df, keep, signed in [
    ('표 2 (tab1)',  tab1,  2, ()),
    ('표 3 (tab2)',  tab2,  1, ()),
    ('표 5 (tab5)',  tab5,  1, ()),
    ('표 8 (tab8)',  tab8,  1, ()),
    ('표 9 (tab9)',  tab9,  1, ('순수출',)),
    ('표 10 (tab10)', tab10, 1, ()),
]:
    print(f'=== {label} — 억 달러 ===')
    to_eok(df, keep, signed)


=== 표 2 (tab1) — 억 달러 ===
| hs10 | 품목명 | 대미 | 전세계 |
|---|---|---|---|
| 8450200000 | 1회의 세탁 능력이 건조한 섬유제품의 중량으로 10킬로그램을 초과하는 것 | 7.31 | 21.45 |
| 8450900000 | 부분품 | 1.79 | 18.55 |
| 8450110000 | 완전자동 세탁기 | 0.99 | 4.98 |
| 8450120000 | 그 밖의 세탁기(원심탈수기를 내장한 것으로 한정한다) | 0.01 | 0.02 |
| 8450190000 | 기타 | 0.00 | 0.01 |

=== 표 3 (tab2) — 억 달러 ===
| 연도 | 완제품_대미 | 완제품_기타 | 부분품_대미 | 부분품_기타 |
|---|---|---|---|---|
| 2010 | 6.39 | 7.59 | 0.83 | 2.42 |
| 2011 | 6.13 | 7.45 | 0.64 | 3.13 |
| 2012 | 5.25 | 8.24 | 0.59 | 2.89 |
| 2013 | 2.91 | 6.91 | 0.58 | 4.34 |
| 2014 | 1.48 | 6.38 | 0.46 | 5.42 |
| 2015 | 0.90 | 4.85 | 0.48 | 4.78 |
| 2016 | 1.54 | 4.03 | 0.45 | 4.70 |
| 2017 | 3.06 | 3.58 | 0.13 | 3.86 |
| 2018 | 1.56 | 2.96 | 0.22 | 1.83 |
| 2019 | 1.25 | 2.74 | 0.51 | 1.59 |
| 2020 | 1.84 | 1.66 | 0.53 | 2.02 |
| 2021 | 4.87 | 2.07 | 0.67 | 2.04 |
| 2022 | 3.46 | 2.03 | 0.91 | 1.60 |
| 2023 | 2.91 | 2.01 | 1.33 | 1.47 |
| 2024 | 3.35 | 2.15 | 1.11 | 1.53 |
| 2025 | 2.33 | 2.31 | 1.33 | 1.52 |

=